In [2]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 1200 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [6]:
from pprint import pprint

query = "How does oxygen concentration affect flame behavior in microgravity?"
results = hybrid_retriever.search(query, top_k=3)

print("Results type:", type(results))
print("Number of results:", len(results))

if results:
    first = results[0]

    print("\nFirst result type:", type(first))

    if isinstance(first, dict):
        print("Top-level keys:", list(first.keys()))

        for key, value in first.items():
            print(f"\n--- {key} ---")
            print("Type:", type(value))

            if isinstance(value, dict):
                pprint(value)
            elif isinstance(value, str):
                print(value[:1500])
            else:
                pprint(value)

    else:
        pprint(first)

Results type: <class 'list'>
Number of results: 3

First result type: <class 'dict'>
Top-level keys: ['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank']

--- chunk_id ---
Type: <class 'int'>
243

--- rrf_score ---
Type: <class 'float'>
0.01639344262295082

--- dense_rank ---
Type: <class 'int'>
1

--- bm25_rank ---
Type: <class 'NoneType'>
None

--- dense_score ---
Type: <class 'float'>
0.7074596285820007

--- bm25_score ---
Type: <class 'NoneType'>
None

--- hybrid_rank ---
Type: <class 'int'>
1


In [7]:

# Inspect the objects used by your existing retrieval pipeline
print("Retriever type:", type(hybrid_retriever))

print("\nRetriever attributes:")
print([
    name for name in dir(hybrid_retriever)
    if not name.startswith("_")
])


Retriever type: <class 'retrieval.hybrid_retriever.HybridRetriever'>

Retriever attributes:
['bm25_retriever', 'dense_retriever', 'reciprocal_rank_fusion', 'rrf_k', 'search']


In [9]:

print("=== DENSE RETRIEVER ===")
print("Type:", type(hybrid_retriever.dense_retriever))
print("Attributes:", [
    name for name in dir(hybrid_retriever.dense_retriever)
    if not name.startswith("_")
])

print("\n=== BM25 RETRIEVER ===")
print("Type:", type(hybrid_retriever.bm25_retriever))
print("Attributes:", [
    name for name in dir(hybrid_retriever.bm25_retriever)
    if not name.startswith("_")
])


=== DENSE RETRIEVER ===
Type: <class 'retrieval.dense_retriever.DenseRetriever'>
Attributes: ['encode_query', 'index', 'index_file', 'model', 'search']

=== BM25 RETRIEVER ===
Type: <class 'retrieval.bm25_retriever.BM25Retriever'>
Attributes: ['bm25', 'chunks', 'chunks_file', 'search', 'tokenize', 'tokenized_corpus']


In [14]:

bm25_chunks = hybrid_retriever.bm25_retriever.chunks

def inspect_retrieval(query, top_k=2):
    results = hybrid_retriever.search(query, top_k=top_k)

    # Map the actual chunk_id to its original chunk
    chunks_by_id = {
        chunk["chunk_id"]: chunk
        for chunk in bm25_chunks
    }

    print(f"\nQUERY: {query}")
    print("=" * 90)

    for result in results:
        chunk_id = result["chunk_id"]
        chunk = chunks_by_id.get(chunk_id)

        if chunk is None:
            print(f"\nChunk ID {chunk_id}: no matching chunk found")
            continue

        metadata = chunk.get("metadata", {})

        print(f"\nHybrid rank: {result.get('hybrid_rank')}")
        print(f"Chunk ID: {chunk_id}")
        print(f"Experiment: {metadata.get('experiment_id')}")
        print(f"Chunk type: {metadata.get('chunk_type')}")
        print(f"Oxygen (%): {metadata.get('oxygen_concentration_pct')}")
        print(f"Flow velocity (cm/s): {metadata.get('flow_velocity_cm_s')}")
        print(f"Flame spread (mm/s): {metadata.get('flame_spread_rate_mm_s')}")
        print(f"Outcome: {metadata.get('outcome')}")
        print(f"Source: {metadata.get('source')}")
        print("Text preview:", chunk.get("text", "")[:300].replace("\n", " "))


queries = [
    "How does oxygen concentration affect flame behavior in microgravity?",
    "What relationship exists between flow velocity and flame spread rate?",
    "Compare experiments with steady flame spread and extinction.",
    "Which experiments have the highest flame spread rates?"
]

for query in queries:
    inspect_retrieval(query, top_k=2)




QUERY: How does oxygen concentration affect flame behavior in microgravity?

Hybrid rank: 1
Chunk ID: 243
Experiment: EXP-0061
Chunk type: risk_summary
Oxygen (%): 15.27
Flow velocity (cm/s): 25.84
Flame spread (mm/s): None
Outcome: self_extinguished
Source: EXP-0061.md
Text preview: # Experiment EXP-0061  ## Fire Risk  - **Risk Score:** 24.7 - **Risk Level:** medium  ## Experimental Summary  Experiment EXP-0061 was conducted as part of the BASS-II investigation using Nomex.  The experiment used an oxygen concentration of 15.27%, a pressure of 68.6 kPa, and a flow velocity of 25

Hybrid rank: 2
Chunk ID: 15
Experiment: EXP-0004
Chunk type: risk_summary
Oxygen (%): 13.52
Flow velocity (cm/s): 38.63
Flame spread (mm/s): None
Outcome: extinguished
Source: EXP-0004.md
Text preview: # Experiment EXP-0004  ## Fire Risk  - **Risk Score:** 24.9 - **Risk Level:** medium  ## Experimental Summary  Experiment EXP-0004 was conducted as part of the Saffire-III investigation using Teflon.  The exper